# Colab + Git LFS demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/studiorabota/test-dg/blob/main/notebooks/lfs_demo.ipynb)

The **Open in Colab** button only loads this `.ipynb` file — not the rest of the repo. So the first step clones the repo and pulls the Git LFS files into the runtime.

**Private repo?** Add a GitHub token as a Colab secret named `GH_TOKEN` (🔑 icon in the left sidebar) and grant this notebook access.

In [ ]:
# Repo settings (publish.sh fills these in for you)
GITHUB_USER = "studiorabota"
REPO_NAME = "test-dg"
BRANCH = "main"

In [ ]:
import os, subprocess, sys

IN_COLAB = "google.colab" in sys.modules

def sh(cmd):
    subprocess.run(cmd, shell=True, check=True)

if IN_COLAB:
    # git-lfs is usually preinstalled on Colab; install it if not
    sh("git lfs version >/dev/null 2>&1 || (apt-get -qq update && apt-get -qq install -y git-lfs)")
    sh("git lfs install")

    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GH_TOKEN")
    except Exception:
        pass  # no secret set -> fine for public repos

    auth = f"{token}@" if token else ""
    url = f"https://{auth}github.com/{GITHUB_USER}/{REPO_NAME}.git"
    REPO_DIR = f"/content/{REPO_NAME}"

    if not os.path.isdir(REPO_DIR):
        result = subprocess.run(["git", "clone", "--branch", BRANCH, url, REPO_DIR],
                                capture_output=True, text=True)
        if result.returncode != 0:
            # Don't echo the URL: it may contain the token
            raise RuntimeError("Clone failed. Check GITHUB_USER / REPO_NAME, "
                               "or add a GH_TOKEN secret for a private repo.")
    sh(f"git -C {REPO_DIR} lfs pull")
else:
    # Running locally from notebooks/: the repo is one level up
    REPO_DIR = os.path.abspath("..")

print("Repo at:", REPO_DIR)

In [ ]:
# Which files are stored in LFS?
sh(f"git -C {REPO_DIR} lfs ls-files --size")

In [ ]:
from pathlib import Path

data_path = Path(REPO_DIR) / "data" / "wind_speed_field.npy"

# Without LFS you'd get a ~130-byte text pointer instead of the real file
if data_path.read_bytes()[:40].startswith(b"version https://git-lfs"):
    raise RuntimeError("This is an LFS pointer, not the data. Run `git lfs pull` in the repo.")

print(f"{data_path.name}: {data_path.stat().st_size / 1e6:.1f} MB — real LFS content ✔")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

field = np.load(data_path)  # (hour, y, x), m/s
print("shape:", field.shape, "| dtype:", field.dtype)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
im = ax1.imshow(field.mean(axis=0), cmap="viridis", origin="lower")
ax1.set_title("Mean wind speed (synthetic)")
fig.colorbar(im, ax=ax1, label="m/s")

ax2.plot(range(field.shape[0]), field.mean(axis=(1, 2)), marker="o")
ax2.set_title("Domain-mean wind speed per hour")
ax2.set_xlabel("Hour")
ax2.set_ylabel("m/s")
plt.tight_layout()
plt.show()